# UQRoute-TC — APIBank markup alignment evidence

**Scope:** Two saved clean development responses from the Qwen 1.5B APIBank format audit. The outputs use `<tool>` or `<toolcall>` markup. The notebook exports their complete response text, parsed calls, and chosen-token evidence for local alignment validation. No inference is performed.

**Provenance:** The source run is pinned to the Task 1 benchmark split and Qwen 1.5B revision. These cases were identified from the saved development format audit after its outputs were inspected.


## 1. Restore saved development records


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib
import json
import math

drive.mount('/content/drive')
output_dir = Path('/content/drive/MyDrive/UQRoute-TC/pilots/qwen25-15b-xml-json-dev-24-v1')
manifest_path = output_dir / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
assert manifest['benchmark_revision'] == 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
assert manifest['split_sha256'] == '688c1013d8fe9d5c1b454d14d4cbe215659e48adc7bdfd63175580e83e2eb0db'
assert manifest['model_id'] == 'Qwen/Qwen2.5-1.5B-Instruct'
assert manifest['model_revision'] == manifest['tokenizer_revision'] == '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
assert manifest['mode'] == 'single_prompt_tool_calling'
assert manifest['temperature'] == 0.001 and manifest['max_tokens'] == 1024
case_ids = ('apibank__level1_301', 'apibank__level1_378')
assert all(case_id in manifest['case_ids'] for case_id in case_ids)
print('Source run:', output_dir)
print('APIBank development cases:', case_ids)


## 2. Export aligned markup responses


In [ ]:
records = []
for case_id in case_ids:
    record_path = output_dir / 'records' / (hashlib.sha256(case_id.encode()).hexdigest() + '.json')
    row = json.loads(record_path.read_text())
    assert row['id'] == case_id and row['benchmark'] == 'apibank'
    assert row['uqroute']['status'] == 'complete' and row['uqroute']['evidence']['valid']
    response = row['uqroute']['response']
    assert len(response['choices']) == 1
    choice = response['choices'][0]
    raw = row['prediction']['raw_output']
    assert choice['message']['content'] == raw and choice['finish_reason'] == 'stop'
    calls = row['prediction']['tool_calls']
    assert calls, 'The saved parser did not return a call'
    assert '<tool>' in raw or '<toolcall' in raw
    tokens = choice['logprobs']['content']
    assert tokens and all(math.isfinite(token['logprob']) for token in tokens)
    visible_tokens = tokens[:-1] if tokens[-1]['token'] in {'<|im_end|>', '<|eot_id|>'} else tokens
    assert visible_tokens and all(token['bytes'] is not None for token in visible_tokens)
    assert b''.join(bytes(token['bytes']) for token in visible_tokens) == raw.encode('utf-8')
    records.append({
        'benchmark': 'apibank', 'id': case_id, 'raw_output': raw,
        'parsed_calls': calls, 'finish_reason': choice['finish_reason'],
        'chosen_tokens': tokens, 'token_evidence': row['uqroute']['evidence'],
    })
    print(case_id, '| calls:', len(calls), '| visible tokens:', len(visible_tokens),
          '| bytes aligned:', True)

report = {'source_manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
          'benchmark_revision': manifest['benchmark_revision'],
          'split_sha256': manifest['split_sha256'],
          'model_id': manifest['model_id'], 'model_revision': manifest['model_revision'],
          'code_revision': manifest['code_revision'], 'records': records}
export_path = output_dir / 'apibank_markup_alignment_evidence.json'
export_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n')
print('Evidence:', export_path)
files.download(str(export_path))
